# 03 · Attach storms — what a GUI restart costs the sessions on it

Every daemon swap re-resumes each session onto a fresh PTY, and that re-resume
window *is* the squish/broken-bottom symptom. This notebook measures the cost of
that window so the constitution's guarantee — other agents' sessions survive our
restarts — has a number attached instead of an opinion.

⛔ **Never deploy in order to measure this.** The deploy causes the symptom. Read
the attach histogram from a restart that already happened.

Phases, in the order a mount walks them: `terminal_mount/begin` →
`ensure_begin`/`ensure_end` → `bootstrap_spawn_scheduled` → `js_wait_begin` →
`terminal_open_attempt` transitions (`begin` → `first_output` → `ready`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) != "notebooks" else os.getcwd()))
sys.path.insert(0, os.path.abspath("."))
import ytrace_helpers as H

WINDOW = os.environ.get("YGG_NOTEBOOK_WINDOW", "30m")
HOST = H.GUI_HOST
H.describe_source(HOST, WINDOW)

In [ ]:
mounts  = H.tail(HOST, since=WINDOW, category="terminal_mount")
opens   = [r for r in H.tail(HOST, since=WINDOW, category="ui_telemetry")
           if r.get("name") == "terminal_open_attempt"]
surface = H.tail(HOST, since=WINDOW, category="surface_request")

counts = H.Counter(H.probe_key(r) for r in mounts)
print(f"terminal_mount events: {len(mounts)}  open attempts: {len(opens)}  surface requests: {len(surface)}\n")
print(H.table([{"phase": k, "count": v} for k, v in counts.most_common(20)], ["phase", "count"]))

In [ ]:
# The retained-rehydrate family: each `skipped_*` is a distinct reason the
# preserved PTY snapshot was NOT replayed. A skip is usually the SAFE outcome
# (the surface was already live), so a raw count is not a fault count.
retained = {k: v for k, v in counts.items() if "retained_rehydrate" in k}
print(H.table([{"reason": k, "count": v} for k, v in sorted(retained.items(), key=lambda kv: -kv[1])],
              ["reason", "count"]) if retained else "(no retained-rehydrate events in this window)")

In [ ]:
# Open-attempt phase timings. The `timing` object attributes a slow resume to a
# phase instead of leaving it to be guessed from a screenshot.
TIMING_FIELDS = [
    "request_to_surface_mounted_ms",
    "surface_mounted_to_first_output_ms",
    "request_to_first_meaningful_output_ms",
    "first_output_to_ready_ms",
    "request_to_ready_ms",
]
by_phase, transitions = {}, H.Counter()
for r in opens:
    p = r.get("payload") or {}
    if not isinstance(p, dict):
        continue
    transitions[p.get("phase") or p.get("transition") or p.get("state") or "?"] += 1
    timing = p.get("timing")
    if isinstance(timing, dict):
        for f in TIMING_FIELDS:
            if isinstance(timing.get(f), (int, float)):
                by_phase.setdefault(f, []).append(float(timing[f]))

print("open-attempt transitions:", dict(transitions) or "(none)")
rows = []
for f in TIMING_FIELDS:
    st = H.percentiles(by_phase.get(f, []))
    if st["n"]:
        rows.append({"phase": f, "n": st["n"], "p50_ms": st["p50"], "p95_ms": st["p95"], "max_ms": st["max"]})
print()
print(H.table(rows, ["phase", "n", "p50_ms", "p95_ms", "max_ms"]) if rows
      else "(no open attempts carried a timing object in this window)")

In [ ]:
# Histogram of mount activity over time. A restart shows as one dense spike;
# a steady trickle is ordinary session opening.
BUCKET_MS = 30_000
series = H.rate_series(mounts, BUCKET_MS)
if series:
    vals = [v for _, v in series]
    print(f"mount events/s : {H.sparkline(vals)}  max={max(vals):.2f}/s")
    print(f"                 {H.ts(series[0][0])} .. {H.ts(series[-1][0])}")
    peak_bucket, peak = max(series, key=lambda kv: kv[1])
    burst = sum(1 for _, v in series if v > (max(vals) * 0.5))
    print(f"peak {peak:.2f}/s at {H.ts(peak_bucket)}; {burst} bucket(s) above half-peak "
          f"=> {'one dense restart spike' if burst <= 2 else 'sustained churn, not a single restart'}")
else:
    print("(no mount events)")

In [ ]:
# A bootstrap that finds an existing attach lease is the primary evidence for a
# slow resume when paired with a long request_to_surface_mounted_ms.
lease_skips = counts.get("terminal_mount/terminal_bootstrap_existing_lease_skip", 0) \
            + counts.get("terminal_mount/bootstrap_spawn_skipped_inactive_retained_host", 0)
mounted = H.percentiles(by_phase.get("request_to_surface_mounted_ms", []))
print(f"bootstrap lease/inactive skips: {lease_skips}")
print(f"request_to_surface_mounted_ms : {mounted}")

In [ ]:
READY_WARN_MS, READY_FAIL_MS = 3_000.0, 10_000.0
MOUNTED_WARN_MS = 1_500.0

v = H.Verdict("Attach storms across a GUI restart")
ready = H.percentiles(by_phase.get("request_to_ready_ms", []))
v.check("request -> ready p95", ready.get("p95"), f"<= {READY_WARN_MS/1000:.0f} s",
        warn_over=READY_WARN_MS, fail_over=READY_FAIL_MS, detail=f"n={ready.get('n', 0)}")
v.check("request -> surface mounted p95", mounted.get("p95"), f"<= {MOUNTED_WARN_MS/1000:.1f} s",
        warn_over=MOUNTED_WARN_MS, fail_over=MOUNTED_WARN_MS * 4, detail=f"n={mounted.get('n', 0)}")
v.check("bootstrap lease skips", float(lease_skips), "0 with a fast mount",
        warn_over=0, fail_over=20,
        detail="harmless for the same in-flight mount; evidence for a stall when the mount is slow")

failed = transitions.get("latched_failure", 0) + transitions.get("request_failed", 0) \
       + transitions.get("session_failed", 0)
v.check("latched / failed open attempts", float(failed), "0", warn_over=0, fail_over=1)

if not opens:
    v.note(H.UNKNOWN, "open attempts", "none in this window — widen YGG_NOTEBOOK_WINDOW "
           "or point at a window that contains a restart")
v.show()